In [ ]:
import pandas as pd
import os
import glob

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pickle

from utils.Toolbox_results import combined_value_counts, plot_sankley

from utils.Toolbox_results import analyze_experiments_by_epoch, evaluate_extraction_methods

from utils.Toolbox_results import extract_best_clients_by_max_cosine_distance, extract_best_clients_by_ranked_summary_initial, extract_best_clients_by_summary_metrics

In [2]:
exp_id = 'NEW_WINDOW'

folders_experiment = [f for f in glob.glob(os.path.join('results/density_base', f'*{exp_id}*')) if os.path.isdir(f)]
# folders_experiment = [f.replace('___0', '').replace('results/density_base\\', '') for f in folders_experiment]

len(folders_experiment)

909

In [3]:
rows = []

for filename in folders_experiment:
    parts = filename.replace('___0', '').replace('results/density_base\\', '').split('__')

    if len(parts) == 3:
        prefix, window_part, suffix = parts
        # Split the window part at 'NEW_WINDOW'
        before_window, after_window = window_part.split('_NEW_WINDOW_')

        row = {
            'prefix': prefix,
            'before_window': before_window[15:],
            'after_window': after_window,
            'suffix': suffix,
            'path' : filename
        }
        rows.append(row)
    else:
        print(f"Skipping invalid format: {filename}")

# Create DataFrame
df = pd.DataFrame(rows)

df[['window', 'step_size', 'interval']] = df['suffix'].str.split('_', expand = True).astype(int)
df[['com_epoch', 'loc_epoch', 'agg', 'window_check']] = df['before_window'].str.split('_', expand = True).astype(int)

df['step_ratio'] = round(df['step_size']  / df['window'], 3)
df['init_state'] = df['prefix'].str[5:7]
df['final_state'] = df['prefix'].str[-2:]

df['exp_id'] = df['init_state']  + '_' + df['final_state']  + '__' + df['after_window']

df[['A', 'B', 'C', 'D', 'E']] = df['after_window'].str.split('_', expand = True)

df = df[['init_state', 'final_state', 'B', 'C', 'D', 'E', 'agg', 'window',
       'step_size', 'step_ratio', 'path']]


target_cols = ['B', 'C', 'D', 'E']
df['match_init'] = df[target_cols].eq(df['init_state'], axis=0).sum(axis=1)
df['match_final'] = df[target_cols].eq(df['final_state'], axis=0).sum(axis=1)

# Get last character of final_state
df['final_last_char'] = df['final_state'].str[-1]

# Get last character of each target column and compare
df['match_final_last_char'] = (
    df[target_cols].apply(lambda x: x.str[-1], axis=1)
    .eq(df['final_last_char'], axis=0)
    .sum(axis=1)
)

df = df[df['match_final'] == 1]

df_density = df[df['final_state'] != 'ML']

df.head()

,init_state,final_state,B,C,D,E,agg,window,step_size,step_ratio,path,match_init,match_final,final_last_char,match_final_last_char
0,LL,LH,HL,ML,LH,LL,12,12,3,0.250,results/density_base\A_62_LL_LH__LL_HL_ML_LH_L...,1,1,H,1
1,LL,LH,HL,ML,LH,LL,12,12,6,0.500,results/density_base\A_62_LL_LH__LL_HL_ML_LH_L...,1,1,H,1
2,LL,LH,HL,ML,LH,LL,12,12,9,0.750,results/density_base\A_62_LL_LH__LL_HL_ML_LH_L...,1,1,H,1
3,LL,LH,HL,ML,LH,LL,12,3,1,0.333,results/density_base\A_62_LL_LH__LL_HL_ML_LH_L...,1,1,H,1
4,LL,LH,HL,ML,LH,LL,12,3,2,0.667,results/density_base\A_62_LL_LH__LL_HL_ML_LH_L...,1,1,H,1


In [5]:
df_density_P1 = df_density[(df_density['step_ratio'] == .444) | (df_density['step_ratio'] == .333)]

folders_density_P1 = df_density_P1['path'].tolist()

combined_value_counts(df_density_P1['agg'], normalize=False, sort_index=True)

,count,percent
agg,,
4,36,0.333333
6,36,0.333333
12,36,0.333333


In [12]:
df_density_P2 = df_density[(df_density['step_ratio'] != .444) & (df_density['step_ratio'] != .333)]
df_density_P1B = df_density_P2[df_density_P2['agg'] == 4]
df_density_P2 = df_density_P2[df_density_P2['agg'] != 4]

folders_density_P2 = df_density_P2['path'].tolist()

df_density_P2[['step_ratio', 'agg']].value_counts().sort_index()
combined_value_counts(df_density_P2[['step_ratio', 'agg']], normalize=False, sort_index=True)

count   percent
step_ratio agg                 
0.250      2      108  0.134831
           3       18  0.022472
           6       54  0.067416
           12      36  0.044944
0.500      2      108  0.134831
           3       36  0.044944
           6       72  0.089888
           12      54  0.067416
0.667      6       54  0.067416
           12      54  0.067416
0.750      2      108  0.134831
           3        9  0.011236
           6       54  0.067416
           12      36  0.044944

In [19]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pickle

exp_id = 'NEW_WINDOW'

folders_experiment = df['path'].tolist()

periods = [{
    'first_4_months': list(range(0, 5)),
    'first_year': list(range(0, 13)),
    'second_year': list(range(11, 24)),
    'all_periods': list(range(0, 24)),
# }, {
#     'first_4_months': list(range(0, 4)),
#     'first_year': list(range(0, 12)),
#     'all_periods': list(range(0, 24)),
}, {
    'first_4_months': list(range(0, 5)),
    'all_periods': list(range(0, 24)),
}
]

periods_tags = ['Overlapped', 'Targeted']

for skip_tgt in [True, False]:
    for epochs_tgt, ep_tag in zip([[0, 1, 2, 3, 4], None], ['selected_epochs', 'all']):
        for period_tgt, tag in zip(periods, periods_tags):

            results = analyze_experiments_by_epoch(data = folders_experiment, epochs = epochs_tgt,
                                                   periods = period_tgt, skip_middle = skip_tgt)

            possible_exps = set([exp.split('__')[-2] for exp in list(results.keys())])

            setup_dict = {}
            for setup in possible_exps:
                exps_setup = [exp for exp in list(results.keys()) if setup in exp]
                setup_dict[setup] = exps_setup

            extraction_methods = [
                ("max cosine", extract_best_clients_by_max_cosine_distance),
                ("rank cosine_mean", extract_best_clients_by_ranked_summary_initial),
                ("rank cosine_sum", extract_best_clients_by_ranked_summary_initial),
                ("summary", extract_best_clients_by_summary_metrics),
            ]

            results_evaluation, summary_df = evaluate_extraction_methods(setup_dict = setup_dict,
                                                                         results = results,
                                                                         extractors = extraction_methods,
                                                                         periods_list = list(period_tgt.keys()),
                                                                         exp_directory = 'results/density_base\\')

            save_results = {
                'data_analysis' : results,
                'evaluation' : results_evaluation,
                'data_evaluation' : summary_df
            }

            with open(f'results/density_base/traceback/detect__{str(skip_tgt).lower()}_{tag}_{ep_tag}.pkl', 'wb') as f:
                pickle.dump(save_results, f)

In [6]:
# 2. Define the periods

periods = {
    'first_4_months': list(range(0, 5)),
    'first_year': list(range(0, 12)),
    'second_year': list(range(11, 24)),
    'all_periods': list(range(0, 24)),
}


results = analyze_experiments_by_epoch(data = folders_density_P1, epochs = None, periods = periods, skip_middle = False)

possible_exps = set([exp.split('__')[-2] for exp in list(results.keys())])

setup_dict = {}
for setup in possible_exps:
    exps_setup = [exp for exp in list(results.keys()) if setup in exp]
    setup_dict[setup] = exps_setup

extraction_methods = [
    ("max cosine", extract_best_clients_by_max_cosine_distance),
    ("rank cosine_mean", extract_best_clients_by_ranked_summary_initial),
    ("rank cosine_sum", extract_best_clients_by_ranked_summary_initial),
    ("summary", extract_best_clients_by_summary_metrics),
]

results_evaluation, summary_df = evaluate_extraction_methods(setup_dict = setup_dict,
                                                             results = results,
                                                             extractors = extraction_methods,
                                                             periods_list = list(periods.keys()),
                                                             exp_directory = 'results/density_base\\')

In [12]:
combined_value_counts(results_evaluation[['period', 'correct_%']], normalize=False, sort_index=True)

count   percent
period         correct_%                 
all_periods    0.0           67  0.023860
               50.0          17  0.006054
               100.0        564  0.200855
all_year       0.0            4  0.001425
               100.0        212  0.075499
first_4_months 0.0           10  0.003561
               50.0           1  0.000356
               100.0        637  0.226852
first_year     0.0           28  0.009972
               50.0           3  0.001068
               100.0        617  0.219729
second_year    0.0          107  0.038105
               50.0          13  0.004630
               100.0        528  0.188034

In [42]:
results_evaluation.head()

processed_evaluation = results_evaluation.copy()
processed_evaluation[['window', 'step_size', 'interval']] = processed_evaluation['setup'].str.split('_', expand = True).astype(int)

processed_evaluation['interval'] = (processed_evaluation['interval'] / 3600).astype(int)
processed_evaluation['step_ratio'] = round(processed_evaluation['step_size']  / processed_evaluation['window'], 3)

processed_evaluation.head()

check,exp_id,period,drift_type,wrong_%,correct_%,setup,method,window,step_size,interval,step_ratio
0,LL_LH__LL_HL_ML_LH_LL,all_periods,first_vs_later,100.0,0.0,9_4_21600,max cosine,9,4,6,0.444
1,LL_LH__LL_HL_ML_LH_LL,all_periods,rolling,0.0,100.0,9_4_21600,max cosine,9,4,6,0.444
2,LL_LH__LL_HL_ML_LH_LL,first_4_months,first_vs_later,0.0,100.0,9_4_21600,max cosine,9,4,6,0.444
3,LL_LH__LL_HL_ML_LH_LL,first_4_months,rolling,0.0,100.0,9_4_21600,max cosine,9,4,6,0.444
4,LL_LH__LL_HL_ML_LH_LL,first_year,first_vs_later,0.0,100.0,9_4_21600,max cosine,9,4,6,0.444


In [41]:
plot_evaluation = processed_evaluation[['period', 'drift_type', 'window', 'interval', 'step_size', 'correct_%']].copy()

for col in plot_evaluation.columns:
    print(col, plot_evaluation[col].unique())

period ['all_periods' 'first_4_months' 'first_year' 'second_year' 'all_year']
drift_type ['first_vs_later' 'rolling']
window [9 3]
interval [ 6 12  4]
step_size [4 1]
correct_% [  0. 100.  50.]


In [ ]:
plot_evaluation = processed_evaluation[['period', 'drift_type', 'window', 'interval', 'step_size', 'correct_%']].copy()

plot_evaluation = plot_evaluation[plot_evaluation['drift_type'] == 'rolling']
plot_evaluation['correct_%'] = plot_evaluation['correct_%'].astype(int).astype(str)

for col in plot_evaluation.columns:
    plot_evaluation[col] = plot_evaluation[col].astype(str)

cross = plot_sankley(df = plot_evaluation,
                     features_map = {'period': {'palette': 'PRGn',
                                                'order': ['all_periods', 'first_4_months', 'first_year', 'second_year', 'all_year']},
                                     'window': {'palette': 'Oranges',
                                                   'order': ['3', '9']},
                                     'step_size': {'palette': 'Blues',
                                                   'order': ['1', '4']},
                                     'correct_%': {'palette': 'Reds',
                                                        'order': ['0', '50', '100']}
                                     })

In [ ]:
cross = plot_sankley(df = plot_evaluation,
                     features_map = {'period': {'palette': 'PRGn',
                                                'order': ['all_periods', 'first_4_months', 'first_year', 'second_year', 'all_year']},
                                     'window': {'palette': 'Oranges',
                                                   'order': ['3', '9']},
                                     'step_size': {'palette': 'Blues',
                                                   'order': ['1', '4']},
                                     'correct_%': {'palette': 'Reds',
                                                        'order': ['0', '50', '100']}
                                     })